# Gaussian Commutator Gauge Operator, explained

Let us consider a pure Gaussian state, defined by a covariance matrix $\Gamma$ that squares to $-1$: $\Gamma^2=-1$. Let $L$ be a subset of all the modes, which we called patch. `GaussianCGO` projects to $V_L=\mathrm{supp}\,\rho_L$. Here $\rho_L$ is the reduced (usually mixed) Gaussian state by tracing out $\bar{L}$. Practically, it is defined by the sub-block $\Gamma_L$.

The CGO process is about projecting every operator $O$ into $V_L$. The bounds are the two ends of $\min/\max\mathrm{Tr}(\sigma B_L)$ over the states of $V_L$ with $\mathrm{Tr}(\sigma[H_L,A_i])=0$, and they bracket the exact expectation because the state itself is in $V_L$.

$H_L$ is the Hamiltonian of the whole state restricted to the terms whose support lies inside the patch; section 2 varies its spectrum.


In [ ]:
import itertools
import time

import torch

from base import CUDA

from FreeFermion.GfPEPS import Bond, Graph, GfPEPS
from FreeFermion.linalg import RandPureCov
from CGO.gaussian import GaussianCGO,Term

torch.manual_seed(0)

PHYSICAL, ENVIRONMENT = 8, 4

whole = RandPureCov(PHYSICAL + ENVIRONMENT)          # [physical | environment]

cgo = GaussianCGO(whole, list(range(PHYSICAL)))

print(f'whole   {tuple(whole.shape)}, a pure state on the patch and the environment')
print(f'patch   Majoranas {cgo.internal_index}')
fock = 2 ** cgo.modes
print(f'fock    {fock} = 2**(physical/2)')
print(f'dim V_L     {cgo.size} of {fock}, the support of the reduced state on the patch')


## 1. The bounds across the environment

Every `whole` differs in the environment, so the patch block differs too.  `basic` is the eigenvalue range of $B_L$ and `CGO` adds the commutator constraints; `in` says whether the exact expectation lies inside.


In [ ]:
OBSERVABLE:Term = (1.0, [0, 1, 2, 3])
PHYSICAL = 8
print(f'{"environment":>11} {"dim V_L":>8} {"<O>":>11} {"basic":>23} {"in":>5} | CGO 2, 4 (width, in)')
for ENVIRONMENT in (2, 4, 6, 8):
    torch.manual_seed(0)
    whole = RandPureCov(PHYSICAL + ENVIRONMENT)  # [physical | environment]
    obj = GaussianCGO(whole, list(range(PHYSICAL)))
    exact = obj.by_wick(OBSERVABLE)
    low, high = obj.basic_CGO(OBSERVABLE)
    cells = []
    for degree in ( 2, 4):
        lower, upper = obj.CGO(OBSERVABLE, degree=degree, eps=1e-5)
        inside = (lower - 1e-6) <= exact <= (upper + 1e-6)
        cells.append(f'{degree}:{upper - lower:.1e},{"Y" if inside else "N"}')
    print(f'{ENVIRONMENT:>11} {obj.size:>8} {exact:>11.4f} [{low:+.4f},{high:+.4f}] '
          f'{str(low - 1e-9 <= exact <= high + 1e-9):>5} | ' + '  '.join(cells))


## 2. What $H_L$ is

`CGO` needs a Hamiltonian on the patch whose commutators with the family define the constraints.  It takes the Hamiltonian of the whole state, `hamiltonian(covariance)`, and keeps the terms whose support lies inside the patch: $H_L=\sum_{a<b\in L}M_{ab}\,\gamma_a\gamma_b$ with $M$ of shape $(2n,2n)$.  At the flat spectrum, $M=-\Gamma$ for a pure state, so the patch block is $-\Gamma_L$, which is diagonal in the patch's own normal-mode basis and therefore commutes with the reduced state; that is what puts the state itself inside the feasible set.

The spectrum is not fixed by the state: the ground state of a quadratic Hamiltonian fixes only the sign of each energy, so every positive spectrum describes the same state.  Handing `CGO` a different one changes the bound, and a random spectrum can cut further than the flat default.  The flat default is also the one that keeps the state itself feasible: it leaves $\max_i|\mathrm{Tr}(\rho_L[H_L,A_i])|$ at $4\times10^{-16}$, while the random spectra below leave $3$ to $6\times10^{-2}$, so those rows show what the knob does rather than a certified bracket.


In [ ]:
PHYSICAL_BIG, ENVIRONMENT_BIG, BOND_BIG = 16, 8, 12

torch.manual_seed(0)
big_sub = RandPureCov(PHYSICAL_BIG + BOND_BIG)
big_environment = RandPureCov(BOND_BIG + ENVIRONMENT_BIG)
big_whole = GfPEPS.from_covariances(
    Graph(2, (Bond(0, 1, BOND_BIG),)), [PHYSICAL_BIG, ENVIRONMENT_BIG],
    [big_sub, big_environment]).contract_all()[0].tensors[0]
big = GaussianCGO(big_whole, list(range(PHYSICAL_BIG)))

system_modes = big.covariance.shape[0] // 2
print(f'whole covariance {tuple(big.covariance.shape)} = {system_modes} modes; '
      f'patch size {big.size} over {big.modes} modes')
exact = big.by_wick(OBSERVABLE)
low, high = big.basic_CGO(OBSERVABLE)
print(f'basic            [{low:+.4f}, {high:+.4f}]  width {high - low:.4f}   <O> = {exact:+.6f}')

torch.manual_seed(1)
spectra = [('flat parent', None)]
spectra += [(f'random positive {k}', 1.0 + torch.rand(system_modes, device=CUDA)) for k in (1, 2)]
for label, spectrum in spectra:
    lower, upper = big.CGO(OBSERVABLE, degree=2, eps=1e-5, energies=spectrum)
    inside = lower - 1e-6 <= exact <= upper + 1e-6
    print(f'{label:18s}[{lower:+.4f}, {upper:+.4f}]  width {upper - lower:.4f}  in={inside}')


## 3. What the solver and the build cost

Two costs sit under the bounds and both grow with the patch.  The first is the solver: `SCS` is a first-order method and its iteration count grows with the accuracy target, so `eps` becomes a real knob once the program is large.  On a physical patch of dimension 128 the same program takes 15 s at `eps=1e-5` and does not converge in 900 s at `1e-9`, while the bound moves by 3e-6 against an interval 1.93 wide.

The second is building the constraints, one `project` call per family member, each materialising a batch of shape $2^p\times\mathrm{size}\times\mathrm{size}$ for $p$ pure modes.  The table times the projection alone, discarding each result; keeping all of them reaches 1.4 GiB at 16 sites.  The same method on a physical Kitaev chain of $N=61$ sites at $\mu=1.9$:


In [ ]:
SITES, MU, T, DELTA = 61, 1.9, 1.0, 1.0

def kitaev_majorana(sites: int, mu: float, t: float, delta: float) -> torch.Tensor:
    '''Majorana matrix of the Kitaev chain, H = (i/4) gamma^T M gamma.'''
    matrix = torch.zeros(2 * sites, 2 * sites, dtype=torch.float64, device=CUDA)
    for i in range(sites):
        matrix[2 * i, 2 * i + 1] = -mu
    for i in range(sites - 1):
        matrix[2 * i, 2 * i + 3] = -t + delta
        matrix[2 * i + 1, 2 * i + 2] = t + delta
    return matrix - matrix.T

def ground_covariance(matrix: torch.Tensor) -> torch.Tensor:
    '''Covariance of the ground state, Gamma = i sign(i M).'''
    values, vectors = torch.linalg.eigh(1j * matrix.to(torch.complex128))
    return (1j * (vectors * torch.sign(values)) @ vectors.conj().T).real.contiguous()

CHAIN_LENGTH = 8
chain_start = (SITES - CHAIN_LENGTH) // 2
chain = GaussianCGO(ground_covariance(kitaev_majorana(SITES, MU, T, DELTA)),
                    list(range(2 * chain_start, 2 * (chain_start + CHAIN_LENGTH))))
chain_observable = (1.0, list(chain.internal_index[:4]))
chain_exact = chain.by_wick(chain_observable)
print(f'{CHAIN_LENGTH} sites: size {chain.size}, {len(chain.mixed)} mixed, {len(chain.pure)} pure, '
      f'{len(list(itertools.combinations(chain.internal_index, 2)))} constraints')
for eps in (1e-6, 1e-5):
    started = time.perf_counter()
    lower, upper = chain.CGO(chain_observable, degree=2, eps=eps)
    elapsed = time.perf_counter() - started
    print(f'  eps={eps:g}: width {upper - lower:.6f} in {elapsed:5.2f} s  '
          f'in={lower - 1e-6 <= chain_exact <= upper + 1e-6}')


In [ ]:
print(f'{"sites":>5} {"size":>5} {"labels":>7} {"project":>9} {"ms/label":>9} {"peak MiB":>9}')
for length in (8, 10, 12, 14):
    start = (SITES - length) // 2
    patch = list(range(2 * start, 2 * (start + length)))
    obj = GaussianCGO(ground_covariance(kitaev_majorana(SITES, MU, T, DELTA)), patch)
    labels = list(itertools.combinations(obj.internal_index, 2))
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    for pair in labels:
        obj.project((1.0, list(pair)))
    elapsed = time.perf_counter() - started
    print(f'{length:>5} {obj.size:>5} {len(labels):>7} {elapsed:>8.2f} s '
          f'{1000 * elapsed / len(labels):>8.2f} {torch.cuda.max_memory_allocated() / 2**20:>8.0f}')


## 4. Notes

* The spanning space is the support of the reduced state on the patch, so the state lies in it by construction and the eigenvalues of $B_L$ bracket the expectation: the `in` column is true everywhere, at every environment and every degree.
* The environment caps the dimension: tracing out $\chi$ Majoranas leaves at most $2^{\chi/2}$ directions, which is why `dim V_L` is 2, 4, 8 and 16 down the table.
* The tightening depends on both the degree and the environment.  At environment 2 neither family cuts; at 4 both cut, to $1.7\times10^{-1}$ from a basic width of $5.2\times10^{-1}$; at 6 both cut, degree 4 to $1.1$; and at 8 only degree 4 cuts, to $1.2$ against a basic width of $2.0$.
* `SCS` at `eps=1e-5` solved every row; `CLARABEL` does not solve programs of this size.  Tightening the tolerance to `1e-9` costs orders of magnitude more time for a change far below the width of the interval.
* The two GPU solvers built alongside `SCS`, cuDSS and CARDAL, are both slower than the CPU backend at these sizes -- 1.6x and 18x on a physical patch of dimension 128.  `profiling/REPORT_sdp_backends.md` has the matrix and the build scripts.
